Comparison of empirical cIM and GNN method for conditional GM prediction at the aggregate level.  
All plots and visualizations are based on validation data.

In [ ]:
from pathlib import Path
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns

import ml_tools as mlt
import sim_ranking as sr

sns.set_theme(style="white")

In [ ]:
gnn_results_dir = "/Users/claudy/dev/work/data/sim_ranking/results/gnn/1216_hp_opt/1219_0853_cv_trial_53"
wdata = "/Users/claudy/dev/work/data"

In [ ]:
# Injected Parameters
gnn_results_dir = "/Users/claudy/dev/work/data/sim_ranking/results/gnn/0317_1033_cv_v4p2NZGMDB_v2p4_4e8s_MagDistFilterFix"


In [ ]:
gnn_results_dir = Path(gnn_results_dir)
cim_results_dir = gnn_results_dir / "cim_results"
warnings.simplefilter(action='ignore', category=FutureWarning)

run_config = sr.ml.gnn_gm.RunConfig.from_yaml(gnn_results_dir / "run_config.yaml")

In [ ]:
# Load observed data to get metadata
nzgmdb_ffp = Path(wdata) / run_config.rel_obs_data_ffp
obs_data = sr.data.load_obs_nzgmdb(nzgmdb_ffp)

In [ ]:
# Load empirical GM params
emp_gm_params_ffp = Path(wdata) / "sim_ranking/emp_gm_params/nzgmdb_v4p2/emp_gm_params_400MaxRjb.parquet"
emp_gm_params = pd.read_parquet(emp_gm_params_ffp)

**Result Directory:** `{python} str(gnn_results_dir)`


In [ ]:
# Load the combined validation results
gnn_val_results = pd.read_parquet(gnn_results_dir / "val_results.parquet").sort_index()
cim_val_results = pd.read_parquet(cim_results_dir / "val_results.parquet").sort_index()

# Load the validation results for each validation fold - GNN
gnn_cv_dirs = [cur_result for cur_result in gnn_results_dir.glob("cv_*") if cur_result.is_dir()]
gnn_cv_val_results = {cur_dir.stem: pd.read_parquet(cur_dir / "val_results.parquet") for cur_dir in gnn_cv_dirs}
gnn_cv_val_res_dfs = {cur_key: sr.ml.gnn_gm.get_residuals(cur_result, ims=run_config.ims) for cur_key, cur_result in gnn_cv_val_results.items()}

gnn_cv_train_results = {cur_dir.stem: pd.read_parquet(cur_dir / "train_results.parquet") for cur_dir in gnn_cv_dirs}

# Load the validation results for each validation fold - cIM
cim_cv_dirs = [cur_dir / "cim_results" for cur_dir in gnn_cv_dirs]
cim_cv_val_results = {cur_dir.parent.stem: pd.read_parquet(cur_dir / "val_results.parquet") for cur_dir in cim_cv_dirs}
cim_cv_val_res_dfs = {cur_key: sr.ml.gnn_gm.get_residuals(cur_result, pred_suffix="cond_mean") for cur_key, cur_result in cim_cv_val_results.items()}

In [ ]:
# Marginal results
mar_val_results = emp_gm_params.copy(deep=True)
mar_val_results = mar_val_results.rename(columns={cur_col: cur_col.replace("_mean", "_pred") for cur_col in mar_val_results.columns if cur_col.endswith("_mean")})
mar_val_results = mar_val_results.rename(columns={cur_col: cur_col.replace("_std_Total", "_pred_std") for cur_col in mar_val_results.columns if cur_col.endswith("_std_Total")})
mar_val_results = mar_val_results.drop(columns=[cur_col for cur_col in mar_val_results.columns if cur_col.endswith("_std_Inter") or cur_col.endswith("_std_Intra")])
mar_val_results = mar_val_results.loc[gnn_val_results.index]
mar_val_results[sr.constants.PSA_KEYS] = gnn_val_results[sr.constants.PSA_KEYS]
mar_val_results["site_int"] = gnn_val_results["site_int"]
# mar_val_results["n_obs_sites"] = gnn_val_results["n_obs_sites"]
mar_val_results["cv_iter"] = gnn_val_results["cv_iter"]

In [ ]:
# Compute marginal residuals
mar_val_res_df = sr.ml.gnn_gm.get_residuals(mar_val_results, ims=sr.constants.PSA_KEYS)

mar_val_res_mean_std_df = pd.concat((mar_val_res_df[sr.constants.PSA_KEYS].mean(axis=0), mar_val_res_df[sr.constants.PSA_KEYS].std(axis=0)), axis=1)
mar_val_res_mean_std_df.columns = ["mean", "std"]

In [ ]:
# Some sanity checking
assert gnn_val_results.shape[0] == cim_val_results.shape[0]
assert gnn_val_results.index.equals(cim_val_results.index)

In [ ]:
# Compute residuals - GNN
gnn_val_res_df = sr.ml.gnn_gm.get_residuals(gnn_val_results, ims=run_config.ims)
assert gnn_val_res_df.index.equals(gnn_val_results.index)
gnn_val_res_df["cv_iter"] = gnn_val_results["cv_iter"]

gnn_val_res_mean_std_df = pd.concat((gnn_val_res_df[run_config.ims].mean(axis=0), gnn_val_res_df[run_config.ims].std(axis=0)), axis=1)
gnn_val_res_mean_std_df.columns = ["mean", "std"]

In [ ]:
# Compute residuals - cIM
cim_val_res_df = sr.ml.gnn_gm.get_residuals(cim_val_results, pred_suffix="cond_mean")
assert cim_val_res_df.index.equals(cim_val_results.index)
cim_val_res_df["cv_iter"] = cim_val_results["cv_iter"]

cim_val_res_mean_std_df = pd.concat((cim_val_res_df[sr.constants.PSA_KEYS].mean(axis=0), cim_val_res_df[sr.constants.PSA_KEYS].std(axis=0)), axis=1)
cim_val_res_mean_std_df.columns = ["mean", "std"]

In [ ]:
# GNN - Compute average and standard deviation of predicted standard deviations as function of period
gnn_pred_std_keys = [f"{cur_key}_pred_std" for cur_key in sr.constants.PSA_KEYS]
gnn_val_pred_std_mean_std_df = pd.concat((gnn_val_results[gnn_pred_std_keys].mean(axis=0), gnn_val_results[gnn_pred_std_keys].std(axis=0)), axis=1)
gnn_val_pred_std_mean_std_df.columns = ["mean", "std"]
gnn_val_pred_std_mean_std_df.index = sr.constants.PSA_KEYS

In [ ]:
# cIM - Compute average and standard deviation of predicted standard deviations as function of period
cim_pred_std_keys = [f"{cur_key}_cond_std" for cur_key in sr.constants.PSA_KEYS]
cim_val_pred_std_mean_std_df = pd.concat((cim_val_results[cim_pred_std_keys].mean(axis=0), cim_val_results[cim_pred_std_keys].std(axis=0)), axis=1)
cim_val_pred_std_mean_std_df.columns = ["mean", "std"]
cim_val_pred_std_mean_std_df.index = sr.constants.PSA_KEYS

In [ ]:
# Marginal - Average predicted standard deviation
marg_pred_std_keys = [f"{cur_key}_pred_std" for cur_key in sr.constants.PSA_KEYS]
mar_val_pred_std_mean_std_df = pd.concat([mar_val_results[marg_pred_std_keys].mean(axis=0), mar_val_results[marg_pred_std_keys].std(axis=0)], axis=1)
mar_val_pred_std_mean_std_df.index = sr.constants.PSA_KEYS
mar_val_pred_std_mean_std_df.columns = ["mean", "std"]

In [ ]:
# Distance matrix
dist_matrix = sr.utils.calculate_distance_matrix(obs_data.sites, obs_data.site_df)

# Correlation Coefficients
corr_data = sr.LBSiteCorrelationData.from_dist_matrix(dist_matrix, run_config.pSA_ims)

In [ ]:
# Plot config
plot_ims = ["pSA_0.01", "pSA_0.1", "pSA_0.5", "pSA_1.0", "pSA_3.0", "pSA_10.0"]
n_plot_ims = len(plot_ims)

## CV Overview

In [ ]:
gnn_cv_val_n_scenarios = pd.DataFrame({cur_key: [cur_cv_res.shape[0], cur_key, "val"] for cur_key, cur_cv_res in gnn_cv_val_results.items()}, index=["n_scenarios", "cv_iter", "type"]).T
gnn_cv_train_n_scenarios = pd.DataFrame({cur_key: [cur_cv_res.shape[0], cur_key, "train"] for cur_key, cur_cv_res in gnn_cv_train_results.items()}, index=["n_scenarios", "cv_iter", "type"]).T

In [ ]:
n_scenarios_df = pd.concat((gnn_cv_val_n_scenarios, gnn_cv_train_n_scenarios), axis=0).sort_values("cv_iter").reset_index(drop=True)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(data=n_scenarios_df[n_scenarios_df["type"] == "train"], x="cv_iter", y="n_scenarios", color="blue", ax=ax1)
ax1.set_title("Number of Training Scenarios")

sns.barplot(data=n_scenarios_df[n_scenarios_df["type"] == "val"], x="cv_iter", y="n_scenarios", color="red", ax=ax2)
ax2.set_title("Number of Validation Scenarios")

fig.tight_layout()

## Residual Histograms

In [ ]:
n_bins = 50
x_min, x_max = -2.5, 2.5
bins = np.linspace(x_min, x_max, n_bins)

fig, axs = mlt.plotting.get_fig_axes(2 * n_plot_ims, 2, -1, (8, 6))

for i, cur_im in enumerate(plot_ims):
    ax1, ax2 = axs[i * 2], axs[i * 2 + 1]
    
    # Empirical results
    cur_mean = cim_val_res_mean_std_df.loc[cur_im, "mean"]
    cur_std = cim_val_res_mean_std_df.loc[cur_im, "std"]
    ax1.axvline(cur_mean, color="k", linestyle="-", label="Mean", linewidth=1.5)
    ax1.axvline(cur_mean + cur_std, color="k", linestyle="--", label="Std", linewidth=1.0)
    ax1.axvline(cur_mean - cur_std, color="k", linestyle="--", linewidth=1.0)
    
    sns.histplot(cim_val_res_df, x=cur_im, bins=bins, ax=ax1, common_bins=True, hue="cv_iter", multiple="stack", legend=False, palette="viridis")
    
    # cur_ax.hist(val_res_df[cur_im], bins=bins, label="Train Data")
    ax1.set_xlabel(f"Residual")
    ax1.set_ylabel(f"Count")
    ax1.grid(linewidth=0.5, alpha=0.5, linestyle="--")
    ax1.set_xlim(x_min, x_max )
    ax1.set_title(f"cIM - {cur_im}")

    ax1.text(0.02, 0.98, f"$\mu = {cur_mean:.2f}$, $\sigma = {cur_std:.2}$", horizontalalignment="left",
                verticalalignment="top", transform=ax1.transAxes)
    
    ax1.text(0.98, 0.98, f"N = {np.count_nonzero(~cim_val_res_df[cur_im].isna())}", horizontalalignment="right",
            verticalalignment="top", transform=ax1.transAxes)
    
    # GNN results
    cur_mean = gnn_val_res_mean_std_df.loc[cur_im, "mean"]
    cur_std = gnn_val_res_mean_std_df.loc[cur_im, "std"]
    ax2.axvline(cur_mean, color="k", linestyle="-", label="Mean", linewidth=1.5)
    ax2.axvline(cur_mean + cur_std, color="k", linestyle="--", label="Std", linewidth=1.0)
    ax2.axvline(cur_mean - cur_std, color="k", linestyle="--", linewidth=1.0)
    
    sns.histplot(gnn_val_res_df, x=cur_im, bins=bins, ax=ax2, common_bins=True, hue="cv_iter", multiple="stack", legend=False, palette="viridis")
    
    # cur_ax.hist(val_res_df[cur_im], bins=bins, label="Train Data")
    ax2.set_xlabel(f"Residual")
    ax2.set_ylabel(f"Count")
    ax2.grid(linewidth=0.5, alpha=0.5, linestyle="--")
    ax2.set_xlim(x_min, x_max )
    ax2.set_title(f"GNN - {cur_im}")

    ax2.text(0.02, 0.98, f"$\mu = {cur_mean:.2f}$, $\sigma = {cur_std:.2}$", horizontalalignment="left",
                verticalalignment="top", transform=ax2.transAxes)
    
    ax2.text(0.98, 0.98, f"N = {np.count_nonzero(~gnn_val_res_df[cur_im].isna())}", horizontalalignment="right",
            verticalalignment="top", transform=ax2.transAxes)
    
fig.tight_layout()

### Bias & (Residual) Standard Deviation

In [ ]:
### CV Bias
# GNN
gnn_cv_val_res_mean_values = pd.concat([cur_cv_val_res[run_config.ims].mean().to_frame(cv_key).T for cv_key, cur_cv_val_res in gnn_cv_val_res_dfs.items()], axis=0)
gnn_cv_val_bias_std = gnn_cv_val_res_mean_values.std(axis=0)

# cIM
cim_cv_val_res_mean_values = pd.concat([cur_cv_val_res[sr.constants.PSA_KEYS].mean().to_frame(cv_key).T for cv_key, cur_cv_val_res in cim_cv_val_res_dfs.items()], axis=0)
cim_cv_val_bias_std = cim_cv_val_res_mean_values.std(axis=0)

### CV Residual Standard Deviation
# GNN
gnn_cv_val_res_std_values = pd.concat([cur_cv_val_res[run_config.ims].std().to_frame(cv_key).T for cv_key, cur_cv_val_res in gnn_cv_val_res_dfs.items()], axis=0)
gnn_cv_val_res_std_std = gnn_cv_val_res_std_values.std(axis=0)

# cIM
cim_cv_val_res_std_values = pd.concat([cur_cv_val_res[sr.constants.PSA_KEYS].std().to_frame(cv_key).T for cv_key, cur_cv_val_res in cim_cv_val_res_dfs.items()], axis=0)
cim_cv_val_res_std_std = cim_cv_val_res_std_values.std(axis=0)

fig, ax1, ax2, ax3, ax4 =  sr.plot_utils.get_bias_residual_fig()

### Bias
# GNN - pSA
ax1.semilogx(sr.constants.PERIODS, gnn_val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"], c="darkblue", linewidth=2.0)
ax1.fill_between(sr.constants.PERIODS, gnn_val_res_mean_std_df.loc[run_config.pSA_ims, "mean"] - gnn_cv_val_bias_std.loc[run_config.pSA_ims], gnn_val_res_mean_std_df.loc[run_config.pSA_ims, "mean"] + gnn_cv_val_bias_std.loc[run_config.pSA_ims], alpha=0.2, color="blue")

# cIM - pSA
ax1.semilogx(sr.constants.PERIODS, cim_val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"], c="r", label="cIM")
ax1.fill_between(sr.constants.PERIODS, cim_val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"] - cim_cv_val_bias_std, cim_val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"] + cim_cv_val_bias_std, alpha=0.2, color="red")

# Marginal - pSA
ax1.semilogx(sr.constants.PERIODS, mar_val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"], c="gray", label="Marginal", linewidth=2.0)

# GNN Non-pSA
if run_config.im_set == sr.constants.IMSet.all:
    ax2.errorbar(run_config.non_pSA_ims, gnn_val_res_mean_std_df.loc[run_config.non_pSA_ims, "mean"], yerr=gnn_cv_val_bias_std.loc[run_config.non_pSA_ims], fmt="o", ms=5, c="darkblue")
    ax2.xaxis.set_tick_params(rotation=90)

### Residual Standard Deviation
# GNN - pSA
ax3.semilogx(sr.constants.PERIODS, gnn_val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "std"], c="darkblue", linewidth=2.0)
ax3.fill_between(sr.constants.PERIODS, gnn_val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "std"] - gnn_cv_val_res_std_std.loc[run_config.pSA_ims], gnn_val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "std"] + gnn_cv_val_res_std_std.loc[run_config.pSA_ims], alpha=0.2, color="blue")
ax3.set_ylim(0.0, 1.5)

# cIM
ax3.semilogx(sr.constants.PERIODS, cim_val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "std"], c="r", linewidth=2.0)
ax3.fill_between(sr.constants.PERIODS, cim_val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "std"] - cim_cv_val_res_std_std, cim_val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "std"] + cim_cv_val_res_std_std, alpha=0.2, color="red")

# Marginal
ax3.semilogx(sr.constants.PERIODS, mar_val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "std"], c="gray", linewidth=2.0)

# GNN - Non-pSA
if run_config.im_set == sr.constants.IMSet.all:
    ax4.errorbar(run_config.non_pSA_ims, gnn_val_res_mean_std_df.loc[run_config.non_pSA_ims, "std"], yerr=gnn_cv_val_res_std_std.loc[run_config.non_pSA_ims], fmt="o", ms=5, c="darkblue")
    ax4.xaxis.set_tick_params(rotation=90)
    ax4.set_ylim(ax3.get_ylim())

In [ ]:
### Predicted Standard Deviation Histograms
# n_bins = 50
# x_min, x_max = 0, 1.5
# bins = np.linspace(x_min, x_max, n_bins)
#
# fig, axs = mlt.plotting.get_fig_axes(2 * n_plot_ims, 2, -1, (8, 6))
#
# for i, cur_im in enumerate(plot_ims):
#     ax1, ax2 = axs[i * 2], axs[i * 2 + 1]
#
#     # Empirical results
#     cur_mean = cim_val_results[f"{cur_im}_cond_std"].mean()
#     cur_std = cim_val_results[f"{cur_im}_cond_std"].std()
#     ax1.axvline(cur_mean, color="k", linestyle="-", label="Mean", linewidth=1.5)
#     ax1.axvline(cur_mean + cur_std, color="k", linestyle="--", label="Std", linewidth=1.0)
#     ax1.axvline(cur_mean - cur_std, color="k", linestyle="--", linewidth=1.0)
#
#     sns.histplot(cim_val_results, x=f"{cur_im}_cond_std", bins=bins, ax=ax1, common_bins=True, hue="cv_iter", multiple="stack", legend=False, palette="viridis")
#
#     # cur_ax.hist(val_res_df[cur_im], bins=bins, label="Train Data")
#     ax1.set_xlabel(f"Predicted Standard Deviation")
#     ax1.set_ylabel(f"Count")
#     ax1.grid(linewidth=0.5, alpha=0.5, linestyle="--")
#     ax1.set_xlim(x_min, x_max )
#     ax1.set_title(f"Empirical - {cur_im}")
#
#     ax1.text(0.02, 0.98, f"$\mu = {cur_mean:.2f}$, $\sigma = {cur_std:.2}$", horizontalalignment="left",
#                 verticalalignment="top", transform=ax1.transAxes)
#
#     # GNN results
#     cur_mean = gnn_val_results[f"{cur_im}_pred_std"].mean()
#     cur_std = gnn_val_results[f"{cur_im}_pred_std"].std()
#     ax2.axvline(cur_mean, color="k", linestyle="-", label="Mean", linewidth=1.5)
#     ax2.axvline(cur_mean + cur_std, color="k", linestyle="--", label="Std", linewidth=1.0)
#     ax2.axvline(cur_mean - cur_std, color="k", linestyle="--", linewidth=1.0)
#
#     sns.histplot(gnn_val_results, x=f"{cur_im}_pred_std", bins=bins, ax=ax2, common_bins=True, hue="cv_iter", multiple="stack", legend=False, palette="viridis")
#
#     # cur_ax.hist(val_res_df[cur_im], bins=bins, label="Train Data")
#     ax2.set_xlabel(f"Predicted Standard Deviation")
#     ax2.set_ylabel(f"Count")
#     ax2.grid(linewidth=0.5, alpha=0.5, linestyle="--")
#     ax2.set_xlim(x_min, x_max )
#     ax2.set_title(f"GNN - {cur_im}")
#
#     ax2.text(0.02, 0.98, f"$\mu = {cur_mean:.2f}$, $\sigma = {cur_std:.2}$", horizontalalignment="left",
#                 verticalalignment="top", transform=ax2.transAxes)
#
# fig.tight_layout()

### Predicted Standard Deviation

In [ ]:
gnn_cv_val_pred_std_mean = np.stack([cur_cv_val_result[gnn_pred_std_keys].mean().values for cur_cv_val_result in gnn_cv_val_results.values()], axis=0)
cim_cv_val_pred_std_mean = np.stack([cur_cv_val_result[cim_pred_std_keys].mean().values for cur_cv_val_result in cim_cv_val_results.values()], axis=0)

fig, ax = plt.subplots(1, 1, figsize=(16, 6))

# GNN
ax.plot(sr.constants.PERIODS, gnn_val_pred_std_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"], label="GNN", marker=".", c="b")
ax.fill_between(sr.constants.PERIODS, gnn_val_pred_std_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"] - gnn_val_pred_std_mean_std_df.loc[sr.constants.PSA_KEYS, "std"], gnn_val_pred_std_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"] + gnn_val_pred_std_mean_std_df.loc[sr.constants.PSA_KEYS, "std"], alpha=0.2, color="b")

# Individual GNN CV residual standard deviations
ax.semilogx(np.asarray(sr.constants.PERIODS)[:, None], gnn_cv_val_pred_std_mean.T, c="b", alpha=0.5, linestyle="--", linewidth=1.0, label=["GNN Individual CV folds"] + [None] * (gnn_cv_val_pred_std_mean.shape[0] - 1))

# cIM
ax.plot(sr.constants.PERIODS, cim_val_pred_std_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"], label="cIM", marker=".", c="r")
ax.fill_between(sr.constants.PERIODS, cim_val_pred_std_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"] - cim_val_pred_std_mean_std_df.loc[sr.constants.PSA_KEYS, "std"], cim_val_pred_std_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"] + cim_val_pred_std_mean_std_df.loc[sr.constants.PSA_KEYS, "std"], alpha=0.2, color="r")

# Individual cIM CV residual standard deviations
ax.semilogx(np.asarray(sr.constants.PERIODS)[:, None], cim_cv_val_pred_std_mean.T, c="red", alpha=0.25, linestyle="--", linewidth=1.0, label=["cIM Individual CV folds"] + [None] * (cim_cv_val_pred_std_mean.shape[0] - 1))

# Marginal
ax.plot(sr.constants.PERIODS, mar_val_pred_std_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"], label="Marginal", marker=".", c="gray")
ax.fill_between(sr.constants.PERIODS, mar_val_pred_std_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"] - mar_val_pred_std_mean_std_df.loc[sr.constants.PSA_KEYS, "std"], mar_val_pred_std_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"] + mar_val_pred_std_mean_std_df.loc[sr.constants.PSA_KEYS, "std"], alpha=0.2, color="gray")

ax.legend()
ax.set_xscale("log")
ax.set_ylim(0.0, 1.5)
ax.set_xlim(0.01, 10)
ax.grid(linewidth=0.5, alpha=0.5, linestyle="--")

fig.tight_layout()

### Predicted Standard Deviation vs. Residual Standard Deviation

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(16, 6))

# GNN - Predicted Standard Deviation
ax.plot(sr.constants.PERIODS, gnn_val_pred_std_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"], label="GNN - Predicted", marker=".", c="navy")
ax.fill_between(sr.constants.PERIODS, gnn_val_pred_std_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"] - gnn_val_pred_std_mean_std_df.loc[sr.constants.PSA_KEYS, "std"], gnn_val_pred_std_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"] + gnn_val_pred_std_mean_std_df.loc[sr.constants.PSA_KEYS, "std"], alpha=0.2, color="navy")

# GNN - Residual Standard Deviation
ax.semilogx(sr.constants.PERIODS, gnn_val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "std"], c="blue", linestyle="--", marker="x", label="GNN - Residual")
ax.fill_between(sr.constants.PERIODS, gnn_val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "std"] - gnn_cv_val_res_std_std[sr.constants.PSA_KEYS], gnn_val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "std"] + gnn_cv_val_res_std_std[sr.constants.PSA_KEYS], alpha=0.2, color="blue", linestyle="--")

# cIM - Predicted Standard Deviation
ax.plot(sr.constants.PERIODS, cim_val_pred_std_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"], label="cIM - Predicted", marker=".", c="r")
ax.fill_between(sr.constants.PERIODS, cim_val_pred_std_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"] - cim_val_pred_std_mean_std_df.loc[sr.constants.PSA_KEYS, "std"], cim_val_pred_std_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"] + cim_val_pred_std_mean_std_df.loc[sr.constants.PSA_KEYS, "std"], alpha=0.2, color="r")

# cIM - Residual Standard Deviation
ax.semilogx(sr.constants.PERIODS, cim_val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "std"], c="orange", linestyle="--", marker="x", label="cIM - Residual")
ax.fill_between(sr.constants.PERIODS, cim_val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "std"] - cim_cv_val_res_std_std, cim_val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "std"] + cim_cv_val_res_std_std, alpha=0.2, color="orange", linestyle="--")

ax.legend()
ax.set_xscale("log")
ax.set_ylim(0.0, 1.5)
ax.set_xlim(0.01, 10)
ax.grid(linewidth=0.5, alpha=0.5, linestyle="--")

fig.tight_layout()

## Trends Magnitude

In [ ]:
gnn_val_res_df["mag"] = obs_data.event_df.loc[gnn_val_res_df.event_id, "mag"].values
gnn_val_results["mag"] = obs_data.event_df.loc[gnn_val_results.event_id, "mag"].values

cim_val_res_df["mag"] = obs_data.event_df.loc[cim_val_res_df.event_id, "mag"].values
cim_val_results["mag"] = obs_data.event_df.loc[cim_val_results.event_id, "mag"].values

In [ ]:
# Magnitude bins
mag_bins = [3, 4.5, 6, 8]
assert gnn_val_res_df.mag.min() >= mag_bins[0] and gnn_val_res_df.mag.max() <= mag_bins[-1]
mag_bin_keys = [f"{mag_bins[i]}_{mag_bins[i+1]}" for i in range(len(mag_bins) - 1)]

gnn_val_res_df["mag_bin"] = pd.cut(gnn_val_res_df.mag, bins=mag_bins, labels=mag_bin_keys)
gnn_val_results["mag_bin"] = pd.cut(gnn_val_results.mag, bins=mag_bins, labels=mag_bin_keys)
gnn_res_mag_groups = gnn_val_res_df.groupby("mag_bin")

cim_val_res_df["mag_bin"] = pd.cut(cim_val_res_df.mag, bins=mag_bins, labels=mag_bin_keys)
cim_val_results["mag_bin"] = pd.cut(cim_val_results.mag, bins=mag_bins, labels=mag_bin_keys)
cim_res_mag_groups = cim_val_res_df.groupby("mag_bin")

### Magnitude Distribution

In [ ]:
# Distribution
fig, ax = plt.subplots(1, 1, figsize=(16, 6))
sns.histplot(gnn_val_res_df.mag, bins=25, ax=ax)
ax.set_title("Magnitude Distribution")
ax.set_xlabel("Magnitude")

for i, (cur_bin, cur_key) in enumerate(zip(mag_bins, mag_bin_keys)):
	ax.axvline(cur_bin, color="k", linestyle="--", linewidth=1.0)
	ax.text((mag_bins[i] + mag_bins[i+1]) / 2, ax.get_ylim()[1] * 0.98, f"N={gnn_res_mag_groups.size()[cur_key]}", horizontalalignment="center", verticalalignment="top")

ax.grid(linewidth=0.5, alpha=0.5, linestyle="--")
ax.set_xlim(3, 8)
fig.tight_layout()

### Magnitude Bias & (Residual) Standard Deviation

In [ ]:
gnn_res_mag_bias = gnn_res_mag_groups[run_config.ims].mean()
gnn_res_mag_bias_std = gnn_res_mag_groups[run_config.ims].std()

cim_res_mag_bias = cim_res_mag_groups[sr.constants.PSA_KEYS].mean()
cim_res_mag_bias_std = cim_res_mag_groups[sr.constants.PSA_KEYS].std()

colors = list(reversed(cm.viridis(np.linspace(0, 1, len(mag_bin_keys)))))

fig, ax1, ax2, ax3, ax4 = sr.plot_utils.get_bias_residual_fig(figsize=(16, 6))

# Bias
for i, (cur_key, cur_group) in enumerate(gnn_res_mag_groups):
    ax1.semilogx(sr.constants.PERIODS, cim_res_mag_bias.loc[cur_key, sr.constants.PSA_KEYS], label=f"cIM" if i == 0 else None, c=colors[i], linestyle="--")
    ax1.semilogx(sr.constants.PERIODS, gnn_res_mag_bias.loc[cur_key, sr.constants.PSA_KEYS], label=f"{cur_key} - N = {gnn_res_mag_groups.size()[cur_key]}", c=colors[i])

ax1.semilogx(sr.constants.PERIODS, gnn_val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"], c="darkblue", linewidth=2)
ax1.fill_between(sr.constants.PERIODS, gnn_val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"] - gnn_cv_val_bias_std[run_config.pSA_ims], gnn_val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"] + gnn_cv_val_bias_std[run_config.pSA_ims], alpha=0.2, color="blue")
ax1.legend()

ax1.semilogx(sr.constants.PERIODS, cim_val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"], c="r", linewidth=2)
ax1.fill_between(sr.constants.PERIODS, cim_val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"] - cim_cv_val_bias_std, cim_val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"] + cim_cv_val_bias_std, alpha=0.2, color="red")

# Bias - Non-pSA
if run_config.im_set == sr.constants.IMSet.all:
    ax2.errorbar(run_config.non_pSA_ims, gnn_val_res_mean_std_df.loc[run_config.non_pSA_ims, "mean"], yerr=gnn_cv_val_bias_std.loc[run_config.non_pSA_ims], fmt="o", ms=5, c="darkblue")
    ax2.xaxis.set_tick_params(rotation=90)
    for i, (cur_key, cur_group) in enumerate(gnn_res_mag_groups):
            ax2.scatter(run_config.non_pSA_ims, gnn_res_mag_bias.loc[cur_key, run_config.non_pSA_ims], color=colors[i])

# Residual Standard Deviation
for i, (cur_key, cur_group) in enumerate(gnn_res_mag_groups):
    ax3.semilogx(sr.constants.PERIODS, gnn_res_mag_bias_std.loc[cur_key, sr.constants.PSA_KEYS], c=colors[i])
    ax3.semilogx(sr.constants.PERIODS, cim_res_mag_bias_std.loc[cur_key, sr.constants.PSA_KEYS], c=colors[i], linestyle="--")

ax3.semilogx(sr.constants.PERIODS, gnn_val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "std"], c="darkblue", linewidth=2)
ax3.fill_between(sr.constants.PERIODS, gnn_val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "std"] - gnn_cv_val_res_std_std[run_config.pSA_ims], gnn_val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "std"] + gnn_cv_val_res_std_std[run_config.pSA_ims], alpha=0.2, color="blue")
ax3.set_ylim(0, 1.5)

ax3.semilogx(sr.constants.PERIODS, cim_val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "std"], c="r", linewidth=2)
ax3.fill_between(sr.constants.PERIODS, cim_val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "std"] - cim_cv_val_res_std_std, cim_val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "std"] + cim_cv_val_res_std_std, alpha=0.2, color="red")


# Residual Standard Deviation - Non-pSA
if run_config.im_set == sr.constants.IMSet.all:
    ax4.errorbar(run_config.non_pSA_ims, gnn_val_res_mean_std_df.loc[run_config.non_pSA_ims, "std"], yerr=gnn_cv_val_res_std_std.loc[run_config.non_pSA_ims], fmt="o", ms=5, c="darkblue")
    ax4.xaxis.set_tick_params(rotation=90)
    for i, (cur_key, cur_group) in enumerate(gnn_res_mag_groups):
        ax4.scatter(run_config.non_pSA_ims, gnn_res_mag_bias_std.loc[cur_key, run_config.non_pSA_ims], color=colors[i])
    ax4.set_ylim(ax3.get_ylim())

## Trends $R_{Rup}$

In [ ]:
gnn_val_res_df["rrup"] = obs_data.record_df.loc[gnn_val_res_df.index, "rrup"].values
gnn_val_results["rrup"] = obs_data.record_df.loc[gnn_val_results.index, "rrup"].values

cim_val_res_df["rrup"] = obs_data.record_df.loc[cim_val_res_df.index, "rrup"].values
cim_val_results["rrup"] = obs_data.record_df.loc[cim_val_results.index, "rrup"].values

In [ ]:
# Distance bins
dist_bins = [0, 25, 50, 100, 225]
assert gnn_val_res_df.rrup.min() >= dist_bins[0] and gnn_val_res_df.rrup.max() <= dist_bins[-1]
dist_bin_keys = [f"{dist_bins[i]}_{dist_bins[i+1]}" for i in range(len(dist_bins) - 1)]

gnn_val_res_df["dist_bin"] = pd.cut(gnn_val_res_df.rrup, bins=dist_bins, labels=dist_bin_keys)
gnn_val_results["dist_bin"] = pd.cut(gnn_val_results.rrup, bins=dist_bins, labels=dist_bin_keys)
gnn_res_dist_groups = gnn_val_res_df.groupby("dist_bin")

cim_val_res_df["dist_bin"] = pd.cut(cim_val_res_df.rrup, bins=dist_bins, labels=dist_bin_keys)
cim_val_results["dist_bin"] = pd.cut(cim_val_results.rrup, bins=dist_bins, labels=dist_bin_keys)
cim_res_dist_groups = cim_val_res_df.groupby("dist_bin")

### $R_{Rup}$ Distribution

In [ ]:
# Distribution
fig, ax = plt.subplots(1, 1, figsize=(16, 6))
sns.histplot(gnn_val_res_df.rrup, bins=25, ax=ax)
ax.set_title("$R_{Rup}$ Distribution")
ax.set_xlabel("$R_{Rup}$")

for i, (cur_bin, cur_key) in enumerate(zip(dist_bins, dist_bin_keys)):
	ax.axvline(cur_bin, color="k", linestyle="--", linewidth=1.0)
	ax.text((dist_bins[i] + dist_bins[i+1]) / 2, ax.get_ylim()[1] * 0.98, f"N={gnn_res_dist_groups.size()[cur_key]}", horizontalalignment="center", verticalalignment="top")

ax.grid(linewidth=0.5, alpha=0.5, linestyle="--")
ax.set_xlim(0, 200)
fig.tight_layout()

### $R_{Rup}$ Bias & (Residual) Standard Deviation

In [ ]:
gnn_res_dist_bias = gnn_res_dist_groups[run_config.ims].mean()
gnn_res_dist_std = gnn_res_dist_groups[run_config.ims].std()

cim_res_dist_bias = cim_res_dist_groups[sr.constants.PSA_KEYS].mean()
cim_res_dist_std = cim_res_dist_groups[sr.constants.PSA_KEYS].std()

colors = list(reversed(cm.viridis(np.linspace(0, 1, len(dist_bin_keys)))))

fig, ax1, ax2, ax3, ax4 = sr.plot_utils.get_bias_residual_fig(figsize=(16, 6))

# Bias
ax1.semilogx(sr.constants.PERIODS, gnn_val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"], c="darkblue", linewidth=2)
ax1.fill_between(sr.constants.PERIODS, gnn_val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"] - gnn_cv_val_bias_std[run_config.pSA_ims], gnn_val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"] + gnn_cv_val_bias_std[run_config.pSA_ims], alpha=0.2, color="blue")

ax1.semilogx(sr.constants.PERIODS, cim_val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"], c="r", linewidth=2)
ax1.fill_between(sr.constants.PERIODS, cim_val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"] - cim_cv_val_bias_std, cim_val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"] + cim_cv_val_bias_std, alpha=0.2, color="red")

for i, (cur_key, cur_group) in enumerate(gnn_res_dist_groups):
    ax1.semilogx(sr.constants.PERIODS, cim_res_dist_bias.loc[cur_key, sr.constants.PSA_KEYS], label=f"cIM" if i == 0 else None, c=colors[i], linestyle="--")
    ax1.semilogx(sr.constants.PERIODS, gnn_res_dist_bias.loc[cur_key, sr.constants.PSA_KEYS], label=f"{cur_key} - N = {gnn_res_dist_groups.size()[cur_key]}", c=colors[i])
ax1.legend(title="Distance Bin")


# Bias - Non-pSA
if run_config.im_set == sr.constants.IMSet.all:
    ax2.errorbar(run_config.non_pSA_ims, gnn_val_res_mean_std_df.loc[run_config.non_pSA_ims, "mean"], yerr=gnn_cv_val_bias_std[run_config.non_pSA_ims], fmt="o", ms=5, c="darkblue")
    ax2.xaxis.set_tick_params(rotation=90)
    for i, (cur_key, cur_group) in enumerate(gnn_res_dist_groups):
            ax2.scatter(run_config.non_pSA_ims, gnn_res_dist_bias.loc[cur_key, run_config.non_pSA_ims], color=colors[i])

# Residual Standard Deviation
ax3.semilogx(sr.constants.PERIODS, gnn_val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "std"], c="darkblue", linewidth=2)
ax3.fill_between(sr.constants.PERIODS, gnn_val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "std"] - gnn_cv_val_res_std_std[run_config.pSA_ims], gnn_val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "std"] + gnn_cv_val_res_std_std[run_config.pSA_ims], alpha=0.2, color="blue")

ax3.semilogx(sr.constants.PERIODS, cim_val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "std"], c="r", linewidth=2)
ax3.fill_between(sr.constants.PERIODS, cim_val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "std"] - cim_cv_val_res_std_std, cim_val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "std"] + cim_cv_val_res_std_std, alpha=0.2, color="red")

for i, (cur_key, cur_group) in enumerate(gnn_res_dist_groups):
    ax3.semilogx(sr.constants.PERIODS, gnn_res_dist_std.loc[cur_key, sr.constants.PSA_KEYS], c=colors[i])
    ax3.semilogx(sr.constants.PERIODS, cim_res_dist_std.loc[cur_key, sr.constants.PSA_KEYS], c=colors[i], linestyle="--")
ax3.set_ylim(0, 1.5)

# Residual Standard Deviation - Non-pSA
if run_config.im_set == sr.constants.IMSet.all:
    ax4.errorbar(run_config.non_pSA_ims, gnn_val_res_mean_std_df.loc[run_config.non_pSA_ims, "std"], yerr=gnn_cv_val_res_std_std[run_config.non_pSA_ims], fmt="o", ms=5, c="darkblue")
    ax4.xaxis.set_tick_params(rotation=90)
    for i, (cur_key, cur_group) in enumerate(gnn_res_dist_groups):
        ax4.scatter(run_config.non_pSA_ims, gnn_res_dist_std.loc[cur_key, run_config.non_pSA_ims], color=colors[i])
    ax4.set_ylim(ax3.get_ylim())

## Trends - Degree of Constraint

In [ ]:
gnn_val_results = sr.utils.compute_degree_of_constraint(gnn_val_results, corr_data)
gnn_val_res_df["doc"] = gnn_val_results["doc"]

## Use DoC from the GNN results
## Note: DoC is different due to different observation site selection method
assert gnn_val_results.index.equals(cim_val_results.index)
assert gnn_val_res_df.index.equals(cim_val_res_df.index)
cim_val_results["doc"] = gnn_val_results["doc"]
cim_val_res_df["doc"] = cim_val_results["doc"]

In [ ]:
# DoC bins
doc_bins = [0, 1.0, 2.25, 3.75, 8.0]
assert gnn_val_res_df.doc.min() >= doc_bins[0] and gnn_val_res_df.doc.max() <= doc_bins[-1]
doc_bin_keys = [f"{doc_bins[i]}_{doc_bins[i + 1]}" for i in range(len(doc_bins) - 1)]

gnn_val_res_df["doc_bin"] = pd.cut(gnn_val_res_df.doc, bins=doc_bins, labels=doc_bin_keys)
gnn_val_results["doc_bin"] = pd.cut(gnn_val_results.doc, bins=doc_bins, labels=doc_bin_keys)
gnn_res_doc_groups = gnn_val_res_df.groupby("doc_bin")

cim_val_res_df["doc_bin"] = gnn_val_res_df["doc_bin"]
cim_val_results["doc_bin"] = gnn_val_results["doc_bin"]
cim_res_doc_groups = cim_val_res_df.groupby("doc_bin")

### Degree of Constraint Distribution

In [ ]:
# Distribution
fig, ax1 = plt.subplots(1, 1, figsize=(16, 6))

sns.histplot(gnn_val_res_df.doc, bins=40, ax=ax1)
ax1.set_xlabel("Degree of Constraint")

for i, (cur_bin, cur_key) in enumerate(zip(doc_bins, doc_bin_keys)):
	ax1.axvline(cur_bin, color="k", linestyle="--", linewidth=1.0)
	ax1.text((doc_bins[i] + doc_bins[i + 1]) / 2, ax1.get_ylim()[1] * 0.98, f"N={gnn_res_doc_groups.size()[cur_key]}", horizontalalignment="center", verticalalignment="top")

ax1.grid(linewidth=0.5, alpha=0.5, linestyle="--")
ax1.set_xlim(0, max(doc_bins))

ax1.set_title("Degree of Constraint Distribution")
fig.tight_layout()

In [ ]:
gnn_res_doc_bias = gnn_res_doc_groups[run_config.ims].mean()
gnn_res_doc_std = gnn_res_doc_groups[run_config.ims].std()

cim_res_doc_bias = cim_res_doc_groups[sr.constants.PSA_KEYS].mean()
cim_res_doc_std = cim_res_doc_groups[sr.constants.PSA_KEYS].std()

colors = list(reversed(cm.viridis(np.linspace(0, 1, len(doc_bin_keys)))))

fig, ax1, ax2, ax3, ax4 = sr.plot_utils.get_bias_residual_fig(figsize=(16, 6))

# Bias - pSA
ax1.semilogx(sr.constants.PERIODS, gnn_val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"], c="darkblue", linewidth=2)
ax1.fill_between(sr.constants.PERIODS, gnn_val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"] - gnn_cv_val_bias_std[run_config.pSA_ims], gnn_val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"] + gnn_cv_val_bias_std[run_config.pSA_ims], alpha=0.2, color="blue")

ax1.semilogx(sr.constants.PERIODS, cim_val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"], c="r", linewidth=2)
ax1.fill_between(sr.constants.PERIODS, cim_val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"] - cim_cv_val_bias_std, cim_val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"] + cim_cv_val_bias_std, alpha=0.2, color="red")

for i, (cur_key, cur_group) in enumerate(gnn_res_doc_groups):
    ax1.semilogx(sr.constants.PERIODS, cim_res_doc_bias.loc[cur_key, sr.constants.PSA_KEYS], label=f"cIM" if i == 0 else None, c=colors[i], linestyle="--")
    ax1.semilogx(sr.constants.PERIODS, gnn_res_doc_bias.loc[cur_key, sr.constants.PSA_KEYS], label=f"{cur_key} - N = {gnn_res_doc_groups.size()[cur_key]}", c=colors[i])
ax1.set_title("Bias")
ax1.legend(title="DoC Bins")

# Bias - Non-pSA
if run_config.im_set == sr.constants.IMSet.all:
    ax2.errorbar(run_config.non_pSA_ims, gnn_val_res_mean_std_df.loc[run_config.non_pSA_ims, "mean"], yerr=gnn_cv_val_bias_std[run_config.non_pSA_ims], fmt="o", ms=5, c="darkblue")
    ax2.xaxis.set_tick_params(rotation=90)
    for i, (cur_key, cur_group) in enumerate(gnn_res_doc_groups):
        ax2.scatter(run_config.non_pSA_ims, gnn_res_doc_bias.loc[cur_key, run_config.non_pSA_ims], color=colors[i])

# Residual Standard Deviation - pSA
ax3.semilogx(sr.constants.PERIODS, gnn_val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "std"], c="darkblue", linewidth=2)
ax3.fill_between(sr.constants.PERIODS, gnn_val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "std"] - gnn_cv_val_res_std_std[run_config.pSA_ims], gnn_val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "std"] + gnn_cv_val_res_std_std[run_config.pSA_ims], alpha=0.2, color="blue")

ax3.semilogx(sr.constants.PERIODS, cim_val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "std"], c="r", linewidth=2)
ax3.fill_between(sr.constants.PERIODS, cim_val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "std"] - cim_cv_val_res_std_std, cim_val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "std"] + cim_cv_val_res_std_std, alpha=0.2, color="red")

for i, (cur_key, cur_group) in enumerate(gnn_res_doc_groups):
    ax3.semilogx(sr.constants.PERIODS, gnn_res_doc_std.loc[cur_key, sr.constants.PSA_KEYS], c=colors[i])
    ax3.semilogx(sr.constants.PERIODS, cim_res_doc_std.loc[cur_key, sr.constants.PSA_KEYS], c=colors[i], linestyle="--")
ax3.set_ylim(0, 1.5)

# Residual Standard Deviation - Non-pSA
if run_config.im_set == sr.constants.IMSet.all:
    ax4.errorbar(run_config.non_pSA_ims, gnn_val_res_mean_std_df.loc[run_config.non_pSA_ims, "std"], yerr=gnn_cv_val_res_std_std[run_config.non_pSA_ims], fmt="o", ms=5, c="darkblue")
    ax4.xaxis.set_tick_params(rotation=90)
    for i, (cur_key, cur_group) in enumerate(gnn_res_doc_groups):
        ax4.scatter(run_config.non_pSA_ims, gnn_res_doc_std.loc[cur_key, run_config.non_pSA_ims], color=colors[i])
    ax4.set_ylim(ax3.get_ylim())
